In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)

path_sessions = '../data/raw/palavritas_sessions.csv'
path_attempts = '../data/raw/palavritas_attempts.csv'
path_profile = '../data/raw/user_profile.csv'

try:
    df_sessions = pd.read_csv(path_sessions)
    df_attempts = pd.read_csv(path_attempts)
    df_profile = pd.read_csv(path_profile)
    print("Dados carregados com sucesso.\n")
except FileNotFoundError as e:
    print(f"Erro de arquivo: {e}")

def diagnostico_inicial(df, nome_df):
    print(f"--- Diagnostico: {nome_df} ---")
    print(f"Linhas: {df.shape[0]} | Colunas: {df.shape[1]}")
    nulos = df.isnull().sum()
    nulos_filtrados = nulos[nulos > 0]
    if not nulos_filtrados.empty:
        print("Valores Nulos por Coluna:")
        print(nulos_filtrados)
    else:
        print("Valores Nulos: Nenhum")
    print(f"Linhas Duplicadas: {df.duplicated().sum()}\n")
    print("-" * 50)

if 'df_sessions' in locals():
    diagnostico_inicial(df_sessions, "Sessões")
    diagnostico_inicial(df_attempts, "Tentativas")
    diagnostico_inicial(df_profile, "Perfil")
    display(df_sessions.head())

Dados carregados com sucesso.

--- Diagnostico: Sessões ---
Linhas: 41157 | Colunas: 13
Valores Nulos por Coluna:
result    63
dtype: int64
Linhas Duplicadas: 1198

--------------------------------------------------
--- Diagnostico: Tentativas ---
Linhas: 147270 | Colunas: 5
Valores Nulos: Nenhum
Linhas Duplicadas: 752

--------------------------------------------------
--- Diagnostico: Perfil ---
Linhas: 800 | Colunas: 15
Valores Nulos por Coluna:
age_range       117
city            297
salary_range    193
dtype: int64
Linhas Duplicadas: 0

--------------------------------------------------


,session_id,user_id,word,word_date,attempts,result,time_to_complete_sec,device,session_hour,streak_day,played_next_day,newsletter_open_before_game,active_d30
0,ab38635a07ede4246661,697e9e150e91bb76,TEMPO,2026-02-08,1,win,449,Android,20,2,False,False,False
1,67d40a22cba1a16fd2c5,d7f63dda905adaec,FALHA,2026-05-08,6,lose,232,ios,19,1,True,False,False
2,3635c5707f62ea022423,6220de621fef79b8,FALHA,2026-04-18,3,win,65,iOS,7,2,False,False,False
3,27243d4c7b1669f71ba7,3aa2f6a86d7fe06c,PRATO,2025-12-22,6,lose,404,Android,23,2,False,False,True
4,cd21162dd072f066c160,97af736a7f2aa637,IDEIA,2026-05-28,1,win,222,Android,6,2,False,True,False


In [21]:
import pandas as pd
import numpy as np

path_sessions = '../data/raw/palavritas_sessions.csv'
df_sessions = pd.read_csv(path_sessions)

print("--- Inspecao de Qualidade: palavritas_sessions ---")
print(f"Linhas: {df_sessions.shape[0]} | Colunas: {df_sessions.shape[1]}")
print(f"Duplicadas exatas: {df_sessions.duplicated().sum()}\n")

nulos = df_sessions.isnull().sum()
print("Valores Nulos:")
print(nulos[nulos > 0] if not nulos[nulos > 0].empty else "Nenhum")
print("\n")

print("Anomalias em Colunas Numericas (Valores < 0):")
colunas_numericas = df_sessions.select_dtypes(include=['int64', 'float64']).columns
for col in colunas_numericas:
    negativos = df_sessions[df_sessions[col] < 0]
    if len(negativos) > 0:
        print(f"- '{col}': {len(negativos)} valores negativos")

print("\nAnomalias em Colunas de Texto (Espacos extras):")
colunas_texto = df_sessions.select_dtypes(include=['object']).columns
for col in colunas_texto:
    espacos = df_sessions[df_sessions[col].str.contains(r'^\s|\s$', na=False, regex=True)]
    if len(espacos) > 0:
        print(f"- '{col}': {len(espacos)} linhas com espacos no inicio/fim")

print("\nTipos de Dados Atuais:")
print(df_sessions.dtypes)

--- Inspecao de Qualidade: palavritas_sessions ---
Linhas: 41157 | Colunas: 13
Duplicadas exatas: 1198

Valores Nulos:
result    63
dtype: int64


Anomalias em Colunas Numericas (Valores < 0):
- 'time_to_complete_sec': 30 valores negativos

Anomalias em Colunas de Texto (Espacos extras):

Tipos de Dados Atuais:
session_id                       str
user_id                          str
word                             str
word_date                        str
attempts                       int64
result                           str
time_to_complete_sec           int64
device                           str
session_hour                   int64
streak_day                     int64
played_next_day                 bool
newsletter_open_before_game     bool
active_d30                      bool
dtype: object


/tmp/ipykernel_122434/512785355.py:24: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colunas_texto = df_sessions.select_dtypes(include=['object']).columns


In [22]:
linhas_antes = len(df_sessions)

df_sessions_clean = df_sessions.copy()

colunas_texto = df_sessions_clean.select_dtypes(include=['object']).columns
for col in colunas_texto:
    df_sessions_clean[col] = df_sessions_clean[col].str.strip().str.lower()

df_sessions_clean = df_sessions_clean.drop_duplicates()
df_sessions_clean = df_sessions_clean.dropna(subset=['result'])
df_sessions_clean['time_to_complete_sec'] = df_sessions_clean['time_to_complete_sec'].abs()
df_sessions_clean['word_date'] = df_sessions_clean['word_date'].apply(
    lambda x: pd.to_datetime(x, dayfirst=True) if '/' in str(x) else pd.to_datetime(x)
)

print(df_sessions_clean['device'].unique())
print(f"Linhas antes: {linhas_antes}")
print(f"Linhas depois: {len(df_sessions_clean)}")
print(f"Colunas: {df_sessions_clean.shape[1]}")
print(f"Duplicadas exatas: {df_sessions_clean.duplicated().sum()}")
print(f"Nulos em result: {df_sessions_clean['result'].isnull().sum()}")
print(f"Negativos em time_to_complete_sec: {(df_sessions_clean['time_to_complete_sec'] < 0).sum()}")
print(f"Tipo de word_date: {df_sessions_clean['word_date'].dtype}")

/tmp/ipykernel_122434/3540199888.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colunas_texto = df_sessions_clean.select_dtypes(include=['object']).columns


<StringArray>
['android', 'ios']
Length: 2, dtype: str
Linhas antes: 41157
Linhas depois: 39899
Colunas: 13
Duplicadas exatas: 0
Nulos em result: 0
Negativos em time_to_complete_sec: 0
Tipo de word_date: datetime64[us]


In [23]:
path_attempts = '../data/raw/palavritas_attempts.csv'
df_attempts = pd.read_csv(path_attempts)

print("--- Inspecao de Qualidade: palavritas_attempts ---")
print(f"Linhas: {df_attempts.shape[0]} | Colunas: {df_attempts.shape[1]}")
print(f"Duplicadas exatas: {df_attempts.duplicated().sum()}\n")

nulos = df_attempts.isnull().sum()
print("Valores Nulos:")
print(nulos[nulos > 0] if not nulos[nulos > 0].empty else "Nenhum")
print("\n")

print("Anomalias em Colunas Numericas (Valores < 0):")
colunas_numericas = df_attempts.select_dtypes(include=['int64', 'float64']).columns
for col in colunas_numericas:
    negativos = df_attempts[df_attempts[col] < 0]
    if len(negativos) > 0:
        print(f"- '{col}': {len(negativos)} valores negativos")

print("\nAnomalias em Colunas de Texto (Espacos extras):")
colunas_texto = df_attempts.select_dtypes(include=['object']).columns
for col in colunas_texto:
    espacos = df_attempts[df_attempts[col].str.contains(r'^\s|\s$', na=False, regex=True)]
    if len(espacos) > 0:
        print(f"- '{col}': {len(espacos)} linhas com espacos no inicio/fim")

print("\nTipos de Dados Atuais:")
print(df_attempts.dtypes)

--- Inspecao de Qualidade: palavritas_attempts ---
Linhas: 147270 | Colunas: 5
Duplicadas exatas: 752

Valores Nulos:
Nenhum


Anomalias em Colunas Numericas (Valores < 0):

Anomalias em Colunas de Texto (Espacos extras):

Tipos de Dados Atuais:
session_id             str
attempt_number       int64
guess                  str
correct_letters      int64
correct_positions    int64
dtype: object


/tmp/ipykernel_122434/1770607133.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colunas_texto = df_attempts.select_dtypes(include=['object']).columns


In [24]:
df_attempts_clean = df_attempts.drop_duplicates()

print("--- Resumo da Limpeza: palavritas_attempts ---")
print(f"Linhas originais: {df_attempts.shape[0]}")
print(f"Linhas apos limpeza: {df_attempts_clean.shape[0]}")
print(f"Duplicadas restantes: {df_attempts_clean.duplicated().sum()}")

--- Resumo da Limpeza: palavritas_attempts ---
Linhas originais: 147270
Linhas apos limpeza: 146518
Duplicadas restantes: 0


In [25]:
df_profile_clean = df_profile.copy()

colunas_texto = df_profile_clean.select_dtypes(include=['object']).columns
for col in colunas_texto:
    df_profile_clean[col] = df_profile_clean[col].str.strip().str.lower()

df_profile_clean = df_profile_clean.fillna("nao_informado")

print(f"Linhas: {df_profile_clean.shape[0]} | Colunas: {df_profile_clean.shape[1]}")
print(f"Nulos totais: {df_profile_clean.isnull().sum().sum()}")

Linhas: 800 | Colunas: 15
Nulos totais: 0


/tmp/ipykernel_122434/2112942562.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colunas_texto = df_profile_clean.select_dtypes(include=['object']).columns


In [26]:
import os

os.makedirs('../data/processed', exist_ok=True)

df_sessions_clean.to_csv('../data/processed/palavritas_sessions_silver.csv', index=False)
df_attempts_clean.to_csv('../data/processed/palavritas_attempts_silver.csv', index=False)
df_profile_clean.to_csv('../data/processed/user_profile_silver.csv', index=False)

print("--- Exportação Concluída ---")
print("Tabelas da camada Silver salvas com sucesso em '../data/processed/':")
print("- palavritas_sessions_silver.csv")
print("- palavritas_attempts_silver.csv")
print("- user_profile_silver.csv")

--- Exportação Concluída ---
Tabelas da camada Silver salvas com sucesso em '../data/processed/':
- palavritas_sessions_silver.csv
- palavritas_attempts_silver.csv
- user_profile_silver.csv
